# LILY WAN 2.2 STUDIO — GPU FIX

Run this first diagnostic/fix cell in Kaggle. It detects the actual NVIDIA GPU without relying on a working PyTorch CUDA kernel. If Kaggle supplied a P100 while its stock Torch dropped sm_60 support, it installs a Pascal-compatible PyTorch stack. After the automatic kernel restart, run the cell once more. T4 uses Kaggle's working stack when possible.


In [ ]:
import os, re, subprocess, sys, json, time

def sh(args, check=True):
    p=subprocess.run(args,text=True,capture_output=True)
    print(p.stdout,end='')
    if p.stderr: print(p.stderr,end='')
    if check and p.returncode: raise RuntimeError('command failed: '+' '.join(args))
    return p

print('=== LILY GPU PREFLIGHT ===')
smi=sh(['nvidia-smi','--query-gpu=name,compute_cap,memory.total','--format=csv,noheader'],False)
if smi.returncode:
    raise RuntimeError('Kaggle has not attached an NVIDIA GPU. Settings → Accelerator → GPU, then restart session.')
print('GPU report:',smi.stdout.strip())
first=smi.stdout.splitlines()[0] if smi.stdout.splitlines() else ''
is_p100='P100' in first.upper()
is_t4='T4' in first.upper()

def torch_probe():
    try:
        import torch
        print('Torch:',torch.__version__,'CUDA:',torch.version.cuda,'available:',torch.cuda.is_available())
        if not torch.cuda.is_available(): return False
        print('Torch GPU:',torch.cuda.get_device_name(0),'capability:',torch.cuda.get_device_capability(0))
        x=torch.ones((64,64),device='cuda',dtype=torch.float16)
        y=x@x
        torch.cuda.synchronize()
        print('CUDA kernel probe: PASS',float(y[0,0]))
        return True
    except Exception as e:
        print('CUDA kernel probe: FAIL:',type(e).__name__,str(e))
        return False

ok=torch_probe()
marker='/kaggle/working/.lily_torch_fixed'
if not ok and is_p100:
    if os.path.exists(marker):
        raise RuntimeError('Pascal-compatible Torch was installed but CUDA still failed after restart. Start a fresh Kaggle P100 session and run again.')
    print('\nP100 detected. Kaggle stock Torch cannot execute sm_60 kernels.')
    print('Installing PyTorch 2.5.1 CUDA 11.8, which includes Pascal/sm_60 support...')
    sh([sys.executable,'-m','pip','install','--no-cache-dir','--force-reinstall','torch==2.5.1','torchvision==0.20.1','torchaudio==2.5.1','--index-url','https://download.pytorch.org/whl/cu118'])
    open(marker,'w').write('torch251-cu118-p100')
    print('\nINSTALL COMPLETE. Restarting the Python kernel now. When Kaggle reconnects, RUN THIS CELL AGAIN.')
    time.sleep(2)
    os._exit(0)
elif not ok:
    raise RuntimeError('A GPU is attached, but its Torch CUDA stack cannot execute kernels. Start a fresh Kaggle GPU session. GPU report: '+first)

print('\nGPU STACK READY:', 'P100' if is_p100 else ('T4' if is_t4 else first))
print('The CUDA failure shown in the previous notebook is resolved if you see CUDA kernel probe: PASS.')
print('Next: this file is intentionally reduced to the environment repair while the Wan runtime is rebuilt against this verified stack.')
